# মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক — আজুর ওপেনএআই (উত্তর API)

এই কোড নমুনায়, আপনি **মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক (MAF)** ব্যবহার করবেন একটি সাধারণ এজেন্ট তৈরি করতে যা **আজুর ওপেনএআই** দ্বারা সমর্থিত এবং **উত্তর API** ব্যবহার করবে।

> **স্থানান্তর নোট:** এই নমুনাটি পূর্বে সিম্যান্টিক কার্নেল ও গিটহাব মডেল ব্যবহার করত। এটি মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্কে স্থানান্তর করা হয়েছে, এবং গিটহাব মডেল (অপ্রচলিত, জুলাই ২০২৬ এ অবসর গ্রহণ করা হবে) এর পরিবর্তে আজুর ওপেনএআই ব্যবহার করা হয়েছে, যা উত্তর API সমর্থন করে। MAF-এ `OpenAIChatClient` আজুর ওপেনএআই-এর স্থিতিশীল `/openai/v1/` এন্ডপয়েন্টের জন্য লক্ষ্যে তৈরি এবং ডিফল্টভাবে উত্তর API ব্যবহার করে।

এই নমুনার উদ্দেশ্য হলো সেই ধাপগুলো প্রদর্শন করা যা পরে বিভিন্ন এজেন্টিক প্যাটার্ন বাস্তবায়নে অতিরিক্ত কোড নমুনায় প্রয়োগ করা হবে।


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## প্রয়োজনীয় পাইথন প্যাকেজগুলি আমদানি করুন


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## একটি টুল সংজ্ঞায়িত করা

Microsoft Agent Framework-এ, একটি **টুল** হল একটি সাধারণ Python ফাংশন যা `@tool` দিয়ে ডেকোরেট করা থাকে এবং যেটি এজেন্ট কল করতে পারে। নিচে আমরা একটি টুল সংজ্ঞায়িত করেছি যা একটি র্যান্ডম ছুটির গন্তব্য প্রদান করে এবং আগের গন্তব্যের পুনরাবৃত্তি এড়ায়।


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## এজেন্ট তৈরি করা

এখানে, আমরা `TravelAgent` নামে এজেন্ট তৈরি করছি।

এই উদাহরণে, আমরা খুব সাধারণ নির্দেশাবলী ব্যবহার করেছি। এজেন্টের আচরণ কিভাবে পরিবর্তিত হয় তা দেখতে এই নির্দেশাবলী পরিবর্তন করুন।


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## এজেন্ট চালানো

এখন আমরা এজেন্ট চালাতে পারি। আমরা একটি `AgentSession` তৈরি করি যাতে এজেন্ট টার্নগুলির মধ্যে কথোপকথন মনে রাখতে পারে, তারপর দুটি `user_inputs` পাঠাই। প্রথমটি একটি যাত্রার জন্য অনুরোধ করে; দ্বিতীয়টি বলে যে ব্যবহারকারী প্রস্তাবটি পছন্দ করেনি এবং অন্য একটি চায় — এজেন্ট সেশন ইতিহাস এবং `get_random_destination` টুল ব্যবহার করে উত্তর দেয়।

আপনি এই বার্তাগুলো পরিবর্তন করে দেখতে পারেন এজেন্ট কিভাবে আলাদা প্রতিক্রিয়া দেখায়। উত্তরগুলি **টোকেন-বাই-টোকেন** স্ট্রিম করা হয়।


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
